# 01 · IVP Preprocessing
**Owner: P2 (IVP)** · Smart Shelf Market Basket Analysis

What this notebook does:
1. Shows each photometric preprocessing step (white balance → denoise → CLAHE) with histograms
2. ArUco-based perspective correction of the shelf + 3×3 grid split
3. Controlled degradations (dim / uneven light / noise) and how preprocessing recovers them
4. **Experiment 2 - preprocessing ablation:** mAP of the trained detector on degraded test images, with vs without preprocessing *(needs weights from notebook 02)*

Runs on any laptop (no GPU needed except section 4, which is fine on CPU too).

In [ ]:
# ============================================================
# SETUP - identical in every notebook. Works on Colab, Mac, Windows.
# ============================================================
import os, sys, subprocess
from pathlib import Path

# Colab only: where to get the code from.
#   - GitHub (recommended): set with  python tools/set_repo_url.py <url>
#   - "" : the IVP_Project folder is in Google Drive at MyDrive/IVP_Project
REPO_URL = ""

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if REPO_URL:
        repo = Path("/content/IVP_Project")
        url = REPO_URL
        try:  # private repo: add a GitHub token as Colab secret GITHUB_TOKEN
            from google.colab import userdata
            url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
        except Exception:
            pass
        if repo.exists():
            subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    else:
        repo = Path("/content/drive/MyDrive/IVP_Project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
else:
    repo = Path.cwd()
    while not (repo / "src").exists() and repo != repo.parent:
        repo = repo.parent

sys.path.insert(0, str(repo))
os.chdir(repo)

from src import env
env.summary()

In [ ]:
import cv2, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import preprocess as pp
from src import demo_data, detection
from src.env import DATASETS_DIR, SESSIONS_DIR, DATA_DIR, RESULTS_DIR, RUNS_DIR
from src.viz import show

DATASET = "indian_grocery"          # folder name under DATASETS_DIR (see notebook 02)
dataset_dir = DATASETS_DIR / DATASET

# Sample images: real dataset if downloaded, otherwise demo images
if (dataset_dir / "test" / "images").exists():
    sample_paths = sorted((dataset_dir / "test" / "images").glob("*.jpg"))[:6]
    print(f"Using REAL dataset images from {dataset_dir}")
else:
    demo_dir = DATASETS_DIR / "demo_products"
    if not demo_dir.exists():
        demo_data.make_demo_detection_dataset(demo_dir)
    sample_paths = sorted((demo_dir / "test" / "images").glob("*.jpg"))[:6]
    print("DEMO MODE - real dataset not found, using synthetic images. Run notebook 02 first for real data.")

img = cv2.imread(str(sample_paths[0]))
print("Image shape:", img.shape)

## 1. Photometric preprocessing, step by step

In [ ]:
wb = pp.white_balance(img)
dn = pp.denoise(wb, "bilateral")
cl = pp.apply_clahe(dn)
show([img, wb, dn, cl], ["Original", "+ Gray-world white balance", "+ Bilateral denoise", "+ CLAHE (final)"],
     suptitle="Preprocessing pipeline")

In [ ]:
# Luminance histograms: CLAHE spreads the intensity distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for a, (im, t) in zip(ax, [(img, "Original"), (cl, "After preprocessing")]):
    L = cv2.cvtColor(im, cv2.COLOR_BGR2LAB)[:, :, 0]
    a.hist(L.ravel(), bins=64, color="steelblue")
    a.set_title(f"{t}  (mean={L.mean():.0f}, std={L.std():.0f})"); a.set_xlabel("L (luminance)")
plt.tight_layout(); plt.show()

## 2. ArUco perspective correction + 3×3 grid
Print `tools/aruco_markers.pdf` and stick markers **0 TL, 1 TR, 2 BR, 3 BL** with each marker's
centre on a corner of the 3×3 area. Until the shelf exists, demo shelf photos are generated.

In [ ]:
real_sessions = (SESSIONS_DIR / "sessions.csv").exists()
session_root = SESSIONS_DIR if real_sessions else DATA_DIR / "sessions_demo"
if not real_sessions and not (session_root / "sessions.csv").exists():
    demo_data.make_demo_sessions(session_root)
print("Using", "REAL" if real_sessions else "DEMO", "shelf photos from", session_root)

first = sorted(p for p in session_root.iterdir() if p.is_dir())[0]
shelf = cv2.imread(str(first / "before.jpg"))

markers = pp.detect_markers(shelf)
vis = shelf.copy()
for mid, c in markers.items():
    cv2.polylines(vis, [c.astype(int)], True, (0, 0, 255), 4)
    cv2.putText(vis, f"id {mid}", tuple(c[0].astype(int)), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
print("Markers found:", sorted(markers))

rectified, H = pp.rectify_shelf(shelf)
if rectified is None:
    print("Not all 4 markers visible - check lighting / marker placement.")
else:
    show([vis, pp.draw_grid(rectified)], ["Camera view + detected markers", "Rectified + 3x3 grid"])
    cells = pp.split_grid(rectified)
    show([c for _, _, c in cells], [f"cell {r},{c}" for r, c, _ in cells], cols=9, size=2)

## 3. Controlled degradations - does preprocessing recover them?

In [ ]:
kinds = ["dim", "uneven", "noise", "combined"]
imgs, titles, rows = [], [], []
for k in kinds:
    d = pp.degrade(img, k)
    r = pp.preprocess(d)
    imgs += [d, r]; titles += [f"{k}", f"{k} + preprocess"]
    rows.append({"condition": k, **{f"raw_{m}": v for m, v in pp.image_stats(d).items()},
                 **{f"pre_{m}": v for m, v in pp.image_stats(r).items()}})
show(imgs, titles, cols=4, size=3.8, suptitle="Degradation vs recovery")
pd.DataFrame(rows).round(1)

## 4. Experiment 2 - Preprocessing ablation (mAP)
For each degradation, the test split is copied twice - degraded **raw** and degraded **+ preprocess** -
and the trained detector is evaluated on both. Result goes to `results/ablation_preprocessing.csv`.

Needs trained weights from notebook 02. Set `WEIGHTS` manually to pick a specific run.

In [ ]:
WEIGHTS = detection.latest_run_weights(which="best.pt")   # or e.g. RUNS_DIR / "public_only_..." / "weights/best.pt"
eval_dir = dataset_dir if (dataset_dir / "test" / "images").exists() else DATASETS_DIR / "demo_products"
print("Weights:", WEIGHTS)
print("Test images from:", eval_dir)

In [ ]:
import shutil

def build_condition(kind, with_pre):
    """Copy test split with a degradation (and optional preprocessing) applied."""
    name = f"{kind}_{'pre' if with_pre else 'raw'}"
    out = DATASETS_DIR / "ablation" / name
    if out.exists():
        shutil.rmtree(out)
    (out / "test" / "images").mkdir(parents=True)
    shutil.copytree(eval_dir / "test" / "labels", out / "test" / "labels")
    for p in (eval_dir / "test" / "images").iterdir():
        im = cv2.imread(str(p))
        if im is None:
            continue
        if kind != "clean":
            im = pp.degrade(im, kind)
        if with_pre:
            im = pp.preprocess(im)
        cv2.imwrite(str(out / "test" / "images" / p.name), im)
    names = detection.read_names(eval_dir)
    import yaml
    with open(out / "data_local.yaml", "w") as f:
        yaml.safe_dump({"path": str(out), "train": "test/images", "val": "test/images", "test": "test/images", "names": names}, f)
    return out / "data_local.yaml"

if WEIGHTS is None:
    print("No trained weights yet - run notebook 02 first, then re-run this section.")
else:
    rows = []
    for kind in ["clean", "dim", "uneven", "noise", "combined"]:
        for with_pre in (False, True):
            y = build_condition(kind, with_pre)
            r = detection.evaluate(WEIGHTS, y, tag=f"ablation_{kind}_{'pre' if with_pre else 'raw'}", split="test")
            rows.append({"condition": kind, "preprocessing": with_pre, "mAP50": r["mAP50"],
                         "mAP50_95": r["mAP50_95"], "precision": r["precision"], "recall": r["recall"]})
    ablation = pd.DataFrame(rows)
    ablation.to_csv(RESULTS_DIR / "ablation_preprocessing.csv", index=False)
    display(ablation.pivot(index="condition", columns="preprocessing", values="mAP50").round(3)
            .rename(columns={False: "mAP50 raw", True: "mAP50 + preprocess"}))

In [ ]:
if WEIGHTS is not None:
    piv = ablation.pivot(index="condition", columns="preprocessing", values="mAP50")
    piv.columns = ["raw", "+ preprocess"]
    piv.plot.bar(figsize=(8, 4), rot=0, color=["#bbbbbb", "#2a7de1"])
    plt.ylabel("mAP@50"); plt.title("Experiment 2 - effect of preprocessing under degraded lighting")
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "ablation_preprocessing.png", dpi=150); plt.show()

**For the paper (P2):** report the ablation table and figure; describe CLAHE (clip limit 2.0, 8×8 tiles),
bilateral filtering, gray-world white balance and the ArUco homography. Note that on *clean* images
preprocessing may slightly lower mAP - that's an honest, expected finding (the model was trained on raw images).